# CSDA catalog inventory

Configure `config/areas.json` and add your AOI GeoJSON using the repository README. Set up Earthdata credentials from the terminal with `python scripts/setup_credentials.py`. The optional authentication cell checks your CSDA profile. The inventory combines public STAC item discovery with separately retrieved account entitlements.

In [ ]:
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd().resolve()
repo = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "scripts" / "csda_common.py").is_file()),
    None,
)
if repo is None:
    raise RuntimeError("Open the repository folder in VS Code or start Jupyter there.")
print(f"Kernel: {sys.executable}")

## Check Earthdata authentication

This cell contacts CSDA and prints a short status summary.

In [ ]:
subprocess.run([sys.executable, str(repo / "scripts" / "test_auth.py")], cwd=repo, check=True);

## Run the inventory

This cell queries CSDA metadata and writes the inventory, summary, and pagination audit into the selected output folder.

In [ ]:
config_path = repo / "config" / "areas.json"
output_dir = repo / "outputs"

if not config_path.is_file():
    raise FileNotFoundError("Copy config/areas.example.json to config/areas.json and edit it.")

subprocess.run(
    [sys.executable, str(repo / "scripts" / "inventory.py"),
     "--config", str(config_path), "--output", str(output_dir)],
    cwd=repo,
    check=True,
);

## Review coverage and pagination

A truncated search may omit records. Rerun with a narrower date range or a higher cap if needed.

In [ ]:
import json
from IPython.display import display
import pandas as pd

for area in json.loads(config_path.read_text(encoding="utf-8"))["areas"]:
    area_dir = output_dir / area["id"]
    print(area["name"])
    display(pd.read_json(area_dir / "pagination_audit.json"))
    display(pd.read_csv(area_dir / "inventory.csv", nrows=10))